# 04회차 실습 노트북 — 데이터 분석 개념의 이해

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adminmrobo/fintech_data_analysis/blob/main/notebooks/04_analysis_concepts.ipynb)

해설 페이지: [https://adminmrobo.github.io/fintech_data_analysis/04_analysis_concepts.html](https://adminmrobo.github.io/fintech_data_analysis/04_analysis_concepts.html)

위에서부터 셀을 차례로 실행하세요(Shift+Enter). 데이터는 저장소 `data/` 폴더(fintech-data 복사본)에서 읽으며, 무작위 시드를 42로 고정해 해설 페이지의 대결 숫자와 같은 값이 나옵니다.

글 · 안상선 ((주)M-Robo 대표)

In [1]:
# 준비: 라이브러리와 데이터 불러오기 (Colab에서는 그대로 실행하면 됩니다)
import os, urllib.parse, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)

BASE = 'https://raw.githubusercontent.com/adminmrobo/fintech_data_analysis/main/data/'
def load(path, **kw):
    """저장소의 data/ 폴더에서 CSV를 읽는다. 내 컴퓨터에 저장소가 있으면 그 파일을, 없으면 GitHub에서 읽는다."""
    for root in ('data', '../data'):
        f = os.path.join(root, path)
        if os.path.exists(f):
            return pd.read_csv(f, **kw)
    return pd.read_csv(BASE + urllib.parse.quote(path), **kw)

RS = 42   # 해설 페이지와 같은 숫자가 나오도록 무작위 시드를 고정

## 1. 학습 점수와 시험 점수
거래 횟수·거래 금액 두 칸으로 KNN 분류기를 만들고, k를 바꿔 가며 학습 데이터 점수와 시험 데이터 점수를 비교합니다.

In [2]:
b = load('Revolution/Bank_customers_data.csv')
b['y'] = (b.Attrition_Flag == 'Attrited Customer').astype(int)   # 1 = 이탈 고객
print(b.shape, '이탈 비율:', round(b.y.mean(), 3))
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
X = b[['Total_Trans_Ct', 'Total_Trans_Amt']].values.astype(float)
Xtr, Xte, ytr, yte = train_test_split(X, b.y, test_size=.25, random_state=RS, stratify=b.y)
sc = StandardScaler().fit(Xtr); Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)
rows = []
for k in [1, 3, 5, 9, 15, 31, 51]:
    m = KNeighborsClassifier(k).fit(Xtr, ytr)
    rows.append((k, round(m.score(Xtr, ytr), 3), round(m.score(Xte, yte), 3)))
pd.DataFrame(rows, columns=['k', '학습 점수', '시험 점수'])

(10127, 22) 이탈 비율: 0.161


,k,학습 점수,시험 점수
0,1,0.998,0.872
1,3,0.935,0.893
2,5,0.924,0.897
3,9,0.922,0.904
4,15,0.917,0.908
5,31,0.913,0.905
6,51,0.912,0.905


## 2. 대결 재현: 학습 점수 99.8%의 k=1 vs 시험 점수로 고른 k=15
k=1은 학습 데이터를 거의 외웁니다. 시험 점수는 어떨까요? 위 표에서 두 줄을 비교해 보세요.

In [3]:
plt.plot([r[0] for r in rows], [r[1] for r in rows], 'o-', label='train')
plt.plot([r[0] for r in rows], [r[2] for r in rows], 's-', label='test')
plt.xscale('log'); plt.xlabel('k'); plt.ylabel('accuracy'); plt.legend(); plt.show()

**AI 판단 연습** — 어떤 업체가 "우리 모델 정확도 99.8%"라고 광고합니다. 가장 먼저 물어야 할 질문 하나는? (힌트: 어떤 데이터로 잰 점수인가)